# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [1]:
from importlib.util import find_spec

required_modules = ("bs4", "pytest", "requests", "schedule")
missing_modules = [
    module for module in required_modules if find_spec(module) is None
]
if missing_modules:
    raise RuntimeError(
        "Не установлены зависимости: " + ", ".join(missing_modules)
    )
print("Зависимости установлены.")

Зависимости установлены.


In [2]:
from __future__ import annotations

import json
import re
import subprocess
import sys
import time
from pathlib import Path
from typing import Any
from urllib.parse import urljoin

import requests
import schedule
from bs4 import BeautifulSoup


current_directory = Path.cwd().resolve()
project_candidates = (
    current_directory,
    current_directory.parent,
    current_directory / "books_scraper",
    current_directory.parent / "books_scraper",
)
PROJECT_ROOT = next(
    (
        candidate
        for candidate in project_candidates
        if (candidate / "scraper.py").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Не найдена папка проекта books_scraper")

BASE_URL = "https://books.toscrape.com/"
DEFAULT_OUTPUT_PATH = PROJECT_ROOT / "artifacts" / "books_data.txt"
DEFAULT_TIMEOUT = 10.0
SCHEDULE_TAG = "books-scraper"
RATING_MAP = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5,
}

print(f"Корень проекта: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")

Корень проекта: C:\Users\User\Desktop\ДЗ_МФТИ\Python_1211\Books_scraper_project
Python: 3.12.0


## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [3]:
def _new_session() -> requests.Session:
    """Создать HTTP-сессию с понятным идентификатором клиента."""

    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": (
                "MIPT-Python-Homework-Books-Scraper/1.0 "
                "(educational project)"
            )
        }
    )
    return session

def _request_html(
    url: str,
    *,
    session: requests.Session,
    timeout: float,
) -> str:
    """Загрузить HTML или выбросить ошибку с адресом проблемной страницы."""

    try:
        response = session.get(url, timeout=timeout)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise requests.RequestException(
            f"Не удалось загрузить страницу: {url}"
        ) from exc
    if response.apparent_encoding:
        response.encoding = response.apparent_encoding
    return response.text

def _parse_price(price_text: str) -> float:
    """Преобразовать строку вида ``£51.77`` в число ``51.77``."""

    match = re.search(r"\d+(?:[.,]\d+)?", price_text)
    if match is None:
        raise ValueError(f"Не удалось распознать цену: {price_text!r}")
    return float(match.group(0).replace(",", "."))

def _parse_stock_count(availability: str) -> int | None:
    """Извлечь количество доступных экземпляров, если оно указано."""

    match = re.search(r"(\d+)\s+available", availability, flags=re.IGNORECASE)
    return int(match.group(1)) if match else None

def parse_book_html(html: str, book_url: str) -> dict[str, Any]:
    """Разобрать HTML одной книги без сетевого запроса.

    Args:
        html: HTML-код страницы книги.
        book_url: Исходный абсолютный URL страницы.

    Returns:
        Нормализованный словарь с данными книги.

    Raises:
        ValueError: Если обязательные элементы карточки отсутствуют.
    """

    soup = BeautifulSoup(html, "html.parser")
    product_main = soup.select_one(".product_main")
    if product_main is None:
        raise ValueError(f"На странице нет блока product_main: {book_url}")

    title_element = product_main.select_one("h1")
    price_element = product_main.select_one(".price_color")
    availability_element = product_main.select_one(".availability")
    rating_element = product_main.select_one(".star-rating")

    missing = [
        name
        for name, element in (
            ("title", title_element),
            ("price", price_element),
            ("availability", availability_element),
            ("rating", rating_element),
        )
        if element is None
    ]
    if missing:
        raise ValueError(
            f"На странице {book_url} отсутствуют поля: {', '.join(missing)}"
        )

    availability = availability_element.get_text(" ", strip=True)
    rating_classes = set(rating_element.get("class", []))
    rating_word = next(
        (word for word in RATING_MAP if word in rating_classes),
        None,
    )
    if rating_word is None:
        raise ValueError(f"Не удалось определить рейтинг книги: {book_url}")

    description_heading = soup.select_one("#product_description")
    description_element = (
        description_heading.find_next_sibling("p")
        if description_heading is not None
        else None
    )
    description = (
        description_element.get_text(" ", strip=True)
        if description_element is not None
        else None
    )

    product_information = {
        row.select_one("th").get_text(" ", strip=True):
        row.select_one("td").get_text(" ", strip=True)
        for row in soup.select("table.table.table-striped tr")
        if row.select_one("th") is not None
        and row.select_one("td") is not None
    }

    return {
        "url": book_url,
        "title": title_element.get_text(" ", strip=True),
        "price_gbp": _parse_price(price_element.get_text(" ", strip=True)),
        "rating": RATING_MAP[rating_word],
        "availability": availability,
        "stock_count": _parse_stock_count(availability),
        "description": description,
        "product_information": product_information,
    }

def get_book_data(
    book_url: str,
    *,
    session: requests.Session | None = None,
    timeout: float = DEFAULT_TIMEOUT,
) -> dict[str, Any]:
    """Загрузить страницу одной книги и вернуть нормализованные данные.

    Args:
        book_url: Абсолютный URL страницы книги.
        session: Необязательная HTTP-сессия для повторного использования
            соединений и для подмены в тестах.
        timeout: Максимальное время ожидания ответа в секундах.

    Returns:
        Словарь, соответствующий контракту ``parse_book_html``.
    """

    owns_session = session is None
    client = session or _new_session()
    try:
        html = _request_html(book_url, session=client, timeout=timeout)
        return parse_book_html(html, book_url)
    finally:
        if owns_session:
            client.close()

In [4]:
book_url = (
    "https://books.toscrape.com/catalogue/"
    "a-light-in-the-attic_1000/index.html"
)
fixture_path = PROJECT_ROOT / "tests" / "fixtures" / "book_page.html"
book_data = parse_book_html(
    fixture_path.read_text(encoding="utf-8"),
    book_url,
)
book_data

{'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html',
 'title': 'A Light in the Attic',
 'price_gbp': 51.77,
 'rating': 3,
 'availability': 'In stock (22 available)',
 'stock_count': 22,
 'description': 'Учебное описание книги.',
 'product_information': {'UPC': 'a897fe39b1053632',
  'Product Type': 'Books',
  'Price (excl. tax)': '£51.77',
  'Price (incl. tax)': '£51.77',
  'Tax': '£0.00',
  'Availability': 'In stock (22 available)',
  'Number of reviews': '0'}}

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [5]:
def extract_book_links(catalogue_html: str, page_url: str) -> list[str]:
    """Извлечь абсолютные ссылки на книги со страницы каталога."""

    soup = BeautifulSoup(catalogue_html, "html.parser")
    return [
        urljoin(page_url, link["href"])
        for link in soup.select("article.product_pod h3 a[href]")
    ]

def extract_next_page_url(catalogue_html: str, page_url: str) -> str | None:
    """Вернуть адрес следующей страницы каталога либо ``None``."""

    soup = BeautifulSoup(catalogue_html, "html.parser")
    next_link = soup.select_one("li.next a[href]")
    if next_link is None:
        return None
    return urljoin(page_url, next_link["href"])

def save_books(
    books: list[dict[str, Any]],
    output_path: str | Path = DEFAULT_OUTPUT_PATH,
) -> Path:
    """Атомарно сохранить полный список книг как JSON в UTF-8."""

    target = Path(output_path)
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_suffix(f"{target.suffix}.tmp")
    temporary.write_text(
        json.dumps(books, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    temporary.replace(target)
    return target

def load_books(
    input_path: str | Path = DEFAULT_OUTPUT_PATH,
) -> list[dict[str, Any]]:
    """Прочитать ранее сохранённый JSON-артефакт."""

    return json.loads(Path(input_path).read_text(encoding="utf-8"))

def scrape_books(
    is_save: bool = False,
    *,
    base_url: str = BASE_URL,
    output_path: str | Path = DEFAULT_OUTPUT_PATH,
    session: requests.Session | None = None,
    timeout: float = DEFAULT_TIMEOUT,
    request_delay: float = 0.0,
    max_pages: int | None = None,
) -> list[dict[str, Any]]:
    """Собрать книги со всех доступных страниц каталога.

    Args:
        is_save: Сохранить ли итог в ``output_path``.
        base_url: Базовый адрес учебного сайта.
        output_path: Путь к JSON-файлу с расширением ``.txt``.
        session: Необязательная HTTP-сессия.
        timeout: Тайм-аут каждого HTTP-запроса в секундах.
        request_delay: Пауза между запросами к карточкам книг.
        max_pages: Ограничение числа страниц для демонстрации и тестов.
            ``None`` означает полный каталог.

    Returns:
        Список нормализованных словарей книг в порядке обхода страниц.

    Raises:
        ValueError: Если ``max_pages`` меньше единицы.
        RuntimeError: Если сайт образует цикл пагинации.
        requests.RequestException: Если один из запросов завершился ошибкой.
    """

    if max_pages is not None and max_pages < 1:
        raise ValueError("max_pages должен быть положительным числом")

    owns_session = session is None
    client = session or _new_session()
    page_url: str | None = urljoin(base_url, "catalogue/page-1.html")
    seen_pages: set[str] = set()
    books: list[dict[str, Any]] = []
    processed_pages = 0

    try:
        while page_url is not None:
            if page_url in seen_pages:
                raise RuntimeError(f"Обнаружен цикл пагинации: {page_url}")
            seen_pages.add(page_url)

            catalogue_html = _request_html(
                page_url,
                session=client,
                timeout=timeout,
            )
            processed_pages += 1

            for book_url in extract_book_links(catalogue_html, page_url):
                books.append(
                    get_book_data(
                        book_url,
                        session=client,
                        timeout=timeout,
                    )
                )
                if request_delay > 0:
                    time.sleep(request_delay)

            if max_pages is not None and processed_pages >= max_pages:
                break
            page_url = extract_next_page_url(catalogue_html, page_url)
    finally:
        if owns_session:
            client.close()

    if is_save:
        save_books(books, output_path)
    return books

In [6]:
if DEFAULT_OUTPUT_PATH.exists():
    res = load_books(DEFAULT_OUTPUT_PATH)
else:
    res = scrape_books(is_save=True, max_pages=1)

print(type(res), len(res))

<class 'list'> 1000


## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [7]:
def configure_schedule(
    at_time: str = "19:00",
    *,
    output_path: str | Path = DEFAULT_OUTPUT_PATH,
    request_delay: float = 0.05,
) -> schedule.Job:
    """Настроить один ежедневный запуск полной выгрузки."""

    schedule.clear(SCHEDULE_TAG)
    return (
        schedule.every()
        .day.at(at_time)
        .do(
            scrape_books,
            is_save=True,
            output_path=output_path,
            request_delay=request_delay,
        )
        .tag(SCHEDULE_TAG)
    )

def run_scheduler(check_interval: float = 30.0) -> None:
    """Бесконечно выполнять ожидающие задачи планировщика."""

    while True:
        schedule.run_pending()
        time.sleep(check_interval)


job = configure_schedule("19:00", output_path=DEFAULT_OUTPUT_PATH)
print(job)
schedule.clear(SCHEDULE_TAG)

Job(interval=1, unit=days, do=scrape_books, args=(), kwargs={'is_save': True, 'output_path': WindowsPath('C:/Users/User/Desktop/ДЗ_МФТИ/Python_1211/Books_scraper_project/artifacts/books_data.txt'), 'request_delay': 0.05})


## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [8]:
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_scraper.py"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
    check=False,
)
print(test_result.stdout)
if test_result.stderr:
    print(test_result.stderr)
assert test_result.returncode == 0

...........                                                              [100%]
11 passed in 0.22s



## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: Готовили парсер книг, который ежедневно обходит сайт о книгах и собирает записи по ним. Версионирование не делается - задание требует последние актуальные данные - каждый день локальный снэпшот как есть заменяется после парсинга по расписанию.`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: По частям. Изучил структуру сайта, изучил подходы к циклу от инициации до сохранения с промежуточными артефактами и механиками. Построил укрупненно схему потока выполнения задачи и потом на каждом участке отвечал на вопрос какая механика, какая библиотека, какой участок кода. Пытался исключить ошибки с обращением к сайту - таймауты, или проблемы с изменением количества книг к парсину, думал как не загнать парсер в бесконечный цикл. Структура данных (модель) очень важна для парсеров, мне кажется, и поэтому особое внимание уделил отработке нектороых кейсов по содержанию страницы. Например, наличие спеуиальных символов (Café “Pangur Bán”) или отсутсвие каких-то записей. Из принципиальных проверок - правильности данных одной книги, полный обход каталога и сохранения результатов - на мой взгляд, самые важные проверки. Автозапуск реализовал чисто через питон - пока запущен, он генерит цикл запросов на обновление, частота 30 сек, возможно избыточна для регулярного использования (раз в день можно делать), но для учебной задачи подходит. Можно потом и в таск менеджере (у меня виндовс) организовать запуск, но это для каких-то регулярных задач может быть востребовано.`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: Несколько способов применения - первое - это подготовка практической методички по механикам задания: (1) "проанализируй предложенные наборы кода к реализации и библиотеки, подготовь общее описание задания и на примерах не из заданий дай на каждый этап подробный разбор механик и дай один учебный пример на каждую механику с подробным разбором". Далее, второе - (2) "Для каждой учебной механики наших из примеров дай подробный анализ возможных автотестов и поясни, зачем они важны и почему так важно это проверять, особое внимание удели ошибкам с структуре данных и критическим рискам на пути выполнения всего процесса, связанных со структурой данных". В рамках магисторского проекта мы делаем парсер для интернет магазинов типа Озон, и я уже имею некоторую насмотренность с проблемами в связи со структурами данных и их запясями. Поэтому я на этом сфокусировал внимание.`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: В ходе выполнения столкнулся с различными развилками в способах его реализации. Например, была развилка - обходить 50 записей, которые были на сайте, или по Next до их исчерпания (выбрал до исчерпания, тк дает устойчивость алгоритма при изменении сайта). Среди прочего, как хранить путь к рабочей папке, как работать с HTTP-запросами - таймауты и ошибки. И тд и тп. много разных ситуативных вопросов, которые нужно учитывать при работе со средой, которая может иметь разные события и ситуации, которые надо отрабатывать. Осторожности потребовало и использование бесконечного цикла через Run All - изюежать зависаний и блокировок выполнений. Возможно, это все очевидные вещи, или, наоборот, лишние детали - считаю полезным для себя проговорить здесь. Кажется, что я избыточно в учебном задании пытался думать о вариативности данных и процессов при исполнении.`

**5. Самооценка по критериям задания**:

`Ответ: Чем больше работаешь над заданием, тем больше нюансов и сомнений, или вопросов из серии "хорошо бы учесть", но формально задание не требовало большого количества усложнение, и учебный сайт выглядит стабильным. Поэтому, думаю, что цебную цель задания выполнил и достиг.`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: Впервые работал с GitHub - это был челиндж. Вроде, все получилось. Вопрос - где граница количеста и аспектов для проверки автотестами в реальных проектах. Задался вопросом... В учебном задании все понятно - не менее 3х. А настоящих проектах? Все критические процесс должны быть с автотестами, получается.`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.

